In [1]:
# Section 1: Setup Env

%pip install pyhealth
# %pip install scikit-learn matplotlib seaborn
%pip install pandas numpy torch scikit-learn



[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 23.2.1 -> 25.0.1
[notice] To update, run: python3.10 -m pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [71]:
# ==========================================
# Step 1: Load MIMIC-III dataset with PyHealth
# ==========================================

from pyhealth.datasets import MIMIC3Dataset
import pandas as pd

DATA_PATH = "../MIMIC3/raw" # CHANGE THIS TO YOUR FOLDER PATH OF THE UNZIPPED FILES

mimic_dataset = MIMIC3Dataset(
    root=DATA_PATH,
    tables=["DIAGNOSES_ICD", "PROCEDURES_ICD", "PRESCRIPTIONS", "LABEVENTS"],
    code_mapping = {"ICD9CM": "CCSCM", "ICD9PROC": "CCSPROC", "NDC": "ATC"},
    dev=True
)


chart_df = pd.read_csv(DATA_PATH + "/CHARTEVENTS.csv", usecols=["SUBJECT_ID", "HADM_ID", "ITEMID", "CHARTTIME", "VALUE"])
d_items = pd.read_csv(DATA_PATH + "/D_ITEMS.csv")
patients_dob_df = pd.read_csv(DATA_PATH + "/PATIENTS.csv", usecols=["SUBJECT_ID", "DOB"], engine="python")

/var/folders/2_/z9nw__lj1zb_qbwcy8742qhw0000gn/T/ipykernel_38629/3244824230.py:18: DtypeWarning: Columns (8) have mixed types. Specify dtype option on import or set low_memory=False.
  chart_df = pd.read_csv(DATA_PATH + "/CHARTEVENTS.csv", usecols=["SUBJECT_ID", "HADM_ID", "ITEMID", "CHARTTIME", "VALUE"])


In [ ]:
# Define a function to check if the patient is over 18
from datetime import datetime

def is_over_18(df, id_col, dob_col, target_id):
    # Locate the row matching the target ID
    dob_raw = df.loc[df[id_col] == target_id, dob_col]

    if dob_raw.empty:
        return False  # ID not found

    # Try parsing the DOB
    dob_parsed = pd.to_datetime(dob_raw.values[0], errors='coerce', infer_datetime_format=True)

    if pd.isna(dob_parsed):
        return False  # Could not parse date

    # Calculate age
    today = pd.Timestamp.today()
    age_years = (today - dob_parsed).days // 365
    print("Patient Age: ", age_years)
    
    return age_years >= 18

In [49]:
# Look at the top matches

# Define vital sign types and associated keywords
vital_keywords = {
    "heartrate": ["heart rate"],
    "resprate": ["respiratory rate"],
    "sysbp": ["systolic"],
    "diabp": ["diastolic"],
    "meanbp": ["mean blood pressure", "arterial bp mean", "non invasive bp mean", "bp mean", "blood pressure mean",],
    "spo2": ["spo2", "oxygen saturation"],
    "temperature": ["temperature", "body temperature"]
}

# Function to search for ITEMIDs
def find_itemids(df, keywords):
    itemids = set()
    for kw in keywords:
        matches = df[df["LABEL"].str.contains(kw, case=False, na=False)]
        itemids.update(matches["ITEMID"].astype(str).tolist())
    return itemids

# Build the dictionary
vital_itemids = {
    vital: find_itemids(d_items, keywords)
    for vital, keywords in vital_keywords.items()
}

# Print the result
for vital, itemids in vital_itemids.items():
    print(f"{vital}: {sorted(itemids)}")


# Flatten ITEMIDs to a set
all_vital_itemids = set().union(*vital_itemids.values())

# Filter rows with those ITEMIDs
chart_df = chart_df[chart_df["ITEMID"].astype(str).isin(all_vital_itemids)]
# Convert CHARTTIME to datetime for grouping
chart_df["CHARTTIME"] = pd.to_datetime(chart_df["CHARTTIME"], errors='coerce')
chart_df["VALUE"] = pd.to_numeric(chart_df["VALUE"], errors='coerce')

# Drop NaNs
chart_df.dropna(subset=["VALUE"], inplace=True)

# Map ITEMIDs to vital names
itemid_to_vital = {itemid: vital for vital, ids in vital_itemids.items() for itemid in ids}
chart_df["vital_type"] = chart_df["ITEMID"].astype(str).map(itemid_to_vital)

# Aggregate: mean value per vital per visit
agg_vitals = chart_df.groupby(["SUBJECT_ID", "HADM_ID", "vital_type"])["VALUE"].mean().unstack("vital_type")
agg_vitals.reset_index(inplace=True)


heartrate: ['211', '220045', '220046', '220047', '3494']
resprate: ['220210', '224688', '224689', '224690', '618', '619']
sysbp: ['220050', '220059', '220179', '224167', '225309', '226850', '226852', '227243', '228152', '3313', '3315', '3317', '3319', '3321', '3323', '3325', '442', '455', '480', '482', '484', '492', '51', '6', '666', '6701', '7643']
diabp: ['153', '220051', '220060', '220180', '224643', '225310', '226851', '226853', '227242', '228151', '8364', '8368', '8440', '8441', '8444', '8445', '8446', '8448', '8502', '8503', '8504', '8505', '8506', '8507', '8508', '8555']
meanbp: ['220052', '220181', '224', '224322', '225312', '443', '456', '52', '5731', '6653', '6702']
spo2: ['226253', '228232', '5820', '646', '6719', '8554']
temperature: ['223761', '223762', '224027', '224642', '224674', '226329', '227054', '228242', '591', '597', '645', '676', '677', '678', '679', '8537']


In [70]:
import os
import pickle
import numpy as np
from scipy.io import loadmat
from pyhealth.data import Patient, Visit

# Helper function to get vitals per patient per visit
def get_vitals_for_visit(patient_id, hadm_id):
    row = agg_vitals[(agg_vitals["SUBJECT_ID"] == patient_id) & (agg_vitals["HADM_ID"] == hadm_id)]
    return row.iloc[0].to_dict() if not row.empty else {}

def aki_detection_fn(patient: Patient, time_window=7):
    """Processes a single patient for AKI detection.

    The goal is to detect Acute Kidney Injury (AKI) in patients based on clinical data.
    This function reads patient records, extracts relevant biomarkers, and creates
    epochs to train a classifier on AKI detection (binary classification task: AKI or no AKI).

    Args:
        record: a list of patient records, where each record is a dictionary containing:
            - 'patient_id': unique identifier for the patient
            - 'signal_file': path to the signal (clinical data) file
            - 'label_file': path to the label file indicating AKI diagnosis (ICD-10/SNOMED codes)
            - 'save_to_path': directory to save the output epochs
        epoch_sec: length of each epoch in seconds
        shift: step size for sliding window to create epochs

    Returns:
        samples: list of dictionaries containing:
            - 'patient_id': patient identifier
            - 'visit_id': visit or record identifier
            - 'record_id': unique identifier for the record
            - 'epoch_path': path to the saved epoch pickle file
            - 'label': 1 for AKI, 0 for non-AKI
    """


    # Define AKI-related diagnosis codes
    aki_icd9_codes = {"5845", "5846", "5847", "5848", "5849", "5844"}
    samples = []

    # Only add samples for patients over 18 years old
    patientOver18 = is_over_18(patients_dob_df, id_col="ID", dob_col="DOB", target_id=patient.patient_id)
    if not patientOver18: return []
    
    visitItems = list(patient.visits.items())
    for i, (visit_id, visit) in enumerate(visitItems):
        next_visit = visitItems[i + 1] if i < len(visitItems) - 1 else None
        (next_visit_id, next_visit_obj) = next_visit if next_visit is not None else (None, None)
        # skip visit if age < 18
        # if (patient.birth_datetime).year is not None and (patient.birth_datetime).year < 18:
        #     continue

        conditions = visit.get_code_list("DIAGNOSES_ICD")
        procedures = visit.get_code_list("PROCEDURES_ICD")
        drugs = visit.get_code_list("PRESCRIPTIONS")
        # skip if no data
        if len(conditions) == 0 or len(procedures) == 0 or len(drugs) == 0:
            continue

        # time delta
        delta = 0 if next_visit_obj is None else (next_visit_obj.encounter_time - visit.encounter_time).days # PAPER RECOMMENDING FROM 24-48 HOURS before detected AKI

        # If AKI shows up in next visit within prediction_window, label current visit
        future_conditions = set() if next_visit is None else set(next_visit_obj.get_code_list("DIAGNOSES_ICD"))
        aki_label = 1 if len(aki_icd9_codes & future_conditions) > 0 and delta <= time_window else 0

        vitals = get_vitals_for_visit(patient.patient_id, visit_id)
        print(vitals)

        samples.append({
            "visit_id": visit_id,
            "patient_id": patient.patient_id,
            "conditions": [conditions],
            "procedures": [procedures],
            "drugs": [drugs],
            "label": aki_label,
            "gender": patient.gender,
            **vitals
        })

    return samples
    
mimic_aki_processed_dataset = mimic_dataset.set_task(aki_detection_fn)

Generating samples for aki_detection_fn:   0%|          | 0/1000 [00:00<?, ?it/s]

Generating samples for aki_detection_fn:   0%|          | 0/1000 [00:00<?, ?it/s]


NameError: name 'is_over_18' is not defined

In [66]:

# ==========================================
# Step 3: Define the RNN Model
# ==========================================

from pyhealth.models import RNN
from pyhealth.trainer import Trainer

# Define the RNN model (LSTM-based)
model = RNN(
    dataset=train_dataset,
    feature_keys=["conditions", "procedures", "labs", "prescriptions"],
    label_key="label",
    mode="binary",
    embedding_dim=128,
    hidden_dim=64,
    num_layers=3,
    rnn_type="LSTM"
)

KeyboardInterrupt: 

In [ ]:
# ==========================================
# Step 4: Train and Evaluate the Model
# ==========================================

trainer = Trainer(model=model, device="cuda:0")
trainer.train(
    train_dataloader=train_dataset.get_dataloader(batch_size=64, shuffle=True),
    epochs=10,
    val_dataloader=val_dataset.get_dataloader(batch_size=64, shuffle=False),
    monitor="pr_auc",
)

# Evaluate on test set
trainer.evaluate(test_dataset.get_dataloader(batch_size=64))

In [ ]:
# ==========================================
# Step 5: Extract Model Activations for TCAV
# ==========================================

import torch

def extract_activations(model, dataloader):
    activations, labels = [], []
    model.eval()
    with torch.no_grad():
        for data in dataloader:
            inputs = model.prepare_input(data, model.feature_keys)
            output, hidden = model.model(inputs)
            activations.append(hidden[-1].cpu().numpy())  # last LSTM layer hidden state
            labels.append(data["label"].numpy())
    return activations, labels

train_acts, train_labels = extract_activations(model, train_dataset.get_dataloader(batch_size=64))

In [ ]:


# ==========================================
# Step 6: Build Concept Activation Vectors (CAVs)
# ==========================================

from sklearn.linear_model import LogisticRegression
import numpy as np

# Example concept: "NSAID prescription"
def define_concept(dataset, concept_key="NSAID"):
    concept_labels = []
    for patient in dataset.samples:
        prescriptions = patient["prescriptions"]
        concept_labels.append(1 if concept_key in prescriptions else 0)
    return np.array(concept_labels)

concept_labels = define_concept(train_dataset, "NSAID")


# Train CAV classifier
cav_classifier = LogisticRegression(max_iter=1000)
cav_classifier.fit(np.concatenate(train_acts), concept_labels)

In [ ]:



# ==========================================
# Step 7: Evaluate TCAV
# ==========================================

# Calculate concept influence
coefs = cav_classifier.coef_[0]
concept_importance = np.mean(coefs)
print(f"Concept Importance (NSAIDs): {concept_importance}")

In [ ]:
# ==========================================
# Step 8: Local Explanation for Single Patient
# ==========================================

# Choose a single patient from test set
patient_data = test_dataset.samples[0]
patient_dataloader = test_dataset.get_dataloader(batch_size=1, shuffle=False)
patient_acts, _ = extract_activations(model, patient_dataloader)

# Compute concept alignment
alignment_score = np.dot(patient_acts[0], coefs) / (np.linalg.norm(patient_acts[0]) * np.linalg.norm(coefs))
print(f"Alignment score for concept (NSAIDs) for patient 0: {alignment_score}")


In [ ]:

# ==========================================
# Step 9: Next Steps and Extensions
# ==========================================

# - Experiment with more clinical concepts (infection, gender, etc.)
# - Automate and expand TCAV analysis
# - Visualization of alignment and concept sensitivity scores
